# Real Estate Model Improvement

Feature engineering, feature selection, hyperparameter optimization, and MLflow logging for a CatBoost flat price model.

Outputs were stripped for public release. Re-run the notebook after configuring the required private data access or local artifact paths.


In [ ]:
import os, mlflow
os.environ["MLFLOW_TRACKING_URI"] = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5001")
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("real_estate_pricing")
print("MLflow tracking at:", mlflow.get_tracking_uri())


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(".env")

os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")
os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.getenv("MLFLOW_S3_ENDPOINT_URL", "https://storage.yandexcloud.net")


In [ ]:
import pandas as pd

df = pd.read_csv("../mlflow_server/baseline/train_data.csv")
print(df.shape)
df.head()


In [ ]:
df.info()
df.isna().mean().sort_values(ascending=False).head(10)


In [ ]:
import matplotlib.pyplot as plt
target_col = "price"         
plt.hist(df[target_col], bins=50)
# Plot title removed during public cleanup.


In [ ]:
num_cols = [c for c in df.columns if df[c].dtype != 'object']
df[num_cols].corr()[target_col].sort_values(ascending=False).head(10)


In [ ]:
# See the project README for context.
display(df.describe().T)

# See the project README for context.
import seaborn as sns
missing = df.isna().mean().sort_values(ascending=False).head(15)
plt.figure(figsize=(8,4))
sns.barplot(x=missing.values, y=missing.index, orient='h')
# Plot title removed during public cleanup.
plt.xlabel("share of NaNs")
plt.xlim(0,1)
plt.show()


In [ ]:
corr = df[num_cols].corr()[target_col].abs().sort_values(ascending=False).iloc[1:16]
top_feats = corr.index.tolist() + [target_col]

plt.figure(figsize=(10,8))
sns.heatmap(df[top_feats].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Top 15 feature correlations with price")
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
sns.scatterplot(data=df, x="total_area", y=target_col, alpha=0.3)
plt.yscale("log")
# Plot title removed during public cleanup.
plt.show()


In [ ]:
# See the project README for context.
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="rooms", y=target_col)
plt.yscale("log")
# Plot title removed during public cleanup.
plt.show()

# See the project README for context.
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="building_type_int", y=target_col)
plt.yscale("log")
# Plot title removed during public cleanup.
plt.show()


In [ ]:
import mlflow, tempfile, pathlib, inspect, json, nbformat, os, matplotlib.pyplot as plt
Path = pathlib.Path

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"

# See the project README for context.
with mlflow.start_run(run_name="eda_v1") as run:
    mlflow.log_metrics({"rows": len(df), "n_features": df.shape[1]})

    # See the project README for context.
    temp_dir = Path(tempfile.mkdtemp())
    for i, fignum in enumerate(plt.get_fignums(), 1):
        path = temp_dir / f"fig_{i}.png"
        plt.figure(fignum).savefig(path, bbox_inches="tight")
        mlflow.log_artifact(str(path), artifact_path="figures")

    # See the project README for context.
    nb_path = Path("real_estate_model_improvement.ipynb").resolve()
    mlflow.log_artifact(str(nb_path), artifact_path="notebooks")

    # See the project README for context.
    mlflow.log_artifact("eda_conclusions.md", artifact_path="reports")

print("EDA logged.  run_id ->", run.info.run_id)


In [ ]:
# See the project README for context.
import yaml, pandas as pd, mlflow, os

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5001"))
mlflow.set_experiment("real_estate_pricing")

# See the project README for context.
with open("../mlflow_server/baseline/params.yaml") as fd:      
    p = yaml.safe_load(fd)

target_col = p["train"]["target_col"]          # 'price'
drop_cols  = p["train"]["drop_cols"]           # ['id', 'building_id', 'flat_id', ...]
num_cols   = p["train"]["preprocess"]["num_cols"]
cat_cols   = p["train"]["preprocess"]["cat_cols"]

# See the project README for context.
train_df = pd.read_csv("../mlflow_server/baseline/train_data.csv")
test_df  = pd.read_csv("../mlflow_server/baseline/test_data.csv")

y_train = train_df[target_col]
X_train = train_df.drop(columns=drop_cols + [target_col])

y_test  = test_df[target_col]
X_test  = test_df.drop(columns=drop_cols + [target_col])

print(f"Train shape {X_train.shape},  Test shape {X_test.shape}")


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from catboost import CatBoostRegressor
import numpy as np

# See the project README for context.
class CustomFeatureAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        X["age"]         = 2025 - X["build_year"]
        X["floor_rel"]   = X["floor"] / X["floors_total"]
        X["first_last"]  = ((X["floor"] == 1) | (X["floor"] == X["floors_total"])).astype(int)
        X["area_per_rm"] = X["total_area"] / X["rooms"].replace(0, np.nan)
        return X

# See the project README for context.
num_cols_ext = num_cols + ["age", "floor_rel", "area_per_rm"]
cat_cols_ext = cat_cols + ["first_last"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(with_mean=False), num_cols_ext),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols_ext)
])

# See the project README for context.
cb_params = p["train"]["catboost"].copy()   #   
cb_params["verbose"] = False               #   CatBoost

core_pipe = Pipeline([
    ("feat_add", CustomFeatureAdder()),
    ("prep",     preprocessor),
    ("model",    CatBoostRegressor(**cb_params))
])


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import mlflow

# See the project README for context.
core_pipe.fit(X_train, y_train)
preds = core_pipe.predict(X_test)

rmse = mean_squared_error(y_test, preds, squared=False)
mae  = mean_absolute_error(y_test, preds)
r2   = r2_score(y_test, preds)
print(f"🔄  RMSE: {rmse:,.0f} | MAE: {mae:,.0f} | R2: {r2:.3f}")


In [ ]:
with mlflow.start_run(run_name="featgen_clean") as run:
    # See the project README for context.
    mlflow.log_params(cb_params)
    mlflow.log_params({
        "custom_feats": "age,floor_rel,first_last,area_per_rm",
        "num_cols": len(num_cols_ext),
        "cat_cols": len(cat_cols_ext)
    })
    # Metrics
    mlflow.log_metrics({
        "rmse": rmse,
        "mae":  mae,
        "r2":   r2
    })
    # See the project README for context.
    mlflow.sklearn.log_model(
        sk_model=core_pipe,
        artifact_path="model",
        registered_model_name="real_estate_price_model"
    )

print("Logged to MLflow | run_id:", run.info.run_id)


In [ ]:
# See the project README for context.
import yaml, pandas as pd, numpy as np, mlflow, os
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from catboost import CatBoostRegressor

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5001"))
mlflow.set_experiment("real_estate_pricing")

with open("../mlflow_server/baseline/params.yaml") as fd:
    p = yaml.safe_load(fd)

target_col = p["train"]["target_col"]
drop_cols  = p["train"]["drop_cols"]
num_cols   = p["train"]["preprocess"]["num_cols"]
cat_cols   = p["train"]["preprocess"]["cat_cols"]

train_df = pd.read_csv("../mlflow_server/baseline/train_data.csv")
test_df  = pd.read_csv("../mlflow_server/baseline/test_data.csv")

X_train = train_df.drop(columns=drop_cols + [target_col])
y_train = train_df[target_col]
X_test  = test_df.drop(columns=drop_cols + [target_col])
y_test  = test_df[target_col]


In [ ]:
# See the project README for context.
def add_custom_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["age"]         = 2025 - out["build_year"]
    out["floor_rel"]   = out["floor"] / out["floors_total"]
    out["first_last"]  = ((out["floor"] == 1) |
                          (out["floor"] == out["floors_total"])).astype(int)
    out["area_per_rm"] = out["total_area"] / out["rooms"].replace(0, np.nan)
    return out

X_train_cf = add_custom_features(X_train)
X_test_cf  = add_custom_features(X_test)

num_cols_ext = [c for c in X_train_cf.columns if c not in cat_cols]
cat_cols_ext = cat_cols + ["first_last"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(with_mean=False), num_cols_ext),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols_ext)
])

base_model = CatBoostRegressor(**{**p["train"]["catboost"], "verbose": False})


In [ ]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.pipeline import make_pipeline
import math, time

def rmse_scorer(y_true, y_pred):          
    return -math.sqrt(mean_squared_error(y_true, y_pred))
rmse_score = make_scorer(rmse_scorer, greater_is_better=True)

pipe_no_prep = make_pipeline(base_model)

# ---------- backward --------------------------------------------
sfs_back = SFS(pipe_no_prep,
               k_features=15,
               forward=False,
               floating=False,
               scoring=rmse_score,
               cv=3,
               n_jobs=-1,
               verbose=2)
start = time.time()
sfs_back = sfs_back.fit(X_train_cf, y_train)
print(f"Backward SFS done in {time.time()-start:.1f}s; RMSE cv = {-sfs_back.k_score_:.0f}")


In [ ]:
# ---------- forward ---------------------------------------------
sfs_fwd = SFS(pipe_no_prep,
              k_features=15,
              forward=True,
              floating=False,
              scoring=rmse_score,
              fixed_features=["total_area"],
              cv=3,
              n_jobs=-1,
              verbose=2)
sfs_fwd = sfs_fwd.fit(X_train_cf, y_train)
print(f"Forward  SFS done; RMSE cv = {-sfs_fwd.k_score_:.0f}")

# See the project README for context.
best_feats = (sfs_back if sfs_back.k_score_ > sfs_fwd.k_score_ else sfs_fwd).k_feature_names_
best_feats = list(best_feats)
print("Selected features (n=15):", best_feats)


In [ ]:
# See the project README for context.
X_train_sel = X_train_cf[best_feats]
X_test_sel  = X_test_cf[best_feats]

final_model = CatBoostRegressor(**{**p["train"]["catboost"], "verbose": False})
final_model.fit(X_train_sel, y_train)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
preds = final_model.predict(X_test_sel)
rmse = mean_squared_error(y_test, preds, squared=False)
mae  = mean_absolute_error(y_test, preds)
r2   = r2_score(y_test, preds)
print(f"FINAL | RMSE: {rmse:,.0f} | MAE: {mae:,.0f} | R2: {r2:.3f}")


In [ ]:
# See the project README for context.
import mlflow, json, tempfile, pathlib
with mlflow.start_run(run_name="feat_select_v3") as run:
    mlflow.log_params(p["train"]["catboost"])
    mlflow.log_params({"selector": "SFS", "n_feats": len(best_feats)})
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    tmp = pathlib.Path(tempfile.mkstemp(suffix=".json")[1])
    tmp.write_text(json.dumps(best_feats, indent=2))
    mlflow.log_artifact(str(tmp), artifact_path="selected_features")

    mlflow.catboost.log_model(final_model,
                              artifact_path="model",
                              registered_model_name="real_estate_price_model")

print("Version v3 registered | run_id:", run.info.run_id)


In [ ]:
# See the project README for context.
from copy import deepcopy
X_train_fs = X_train_sel
X_test_fs  = X_test_sel
y_train_fs = y_train
y_test_fs  = y_test


In [ ]:
import optuna, numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

def objective(trial):
    params = deepcopy(p["train"]["catboost"])
    params.update({
        "depth":          trial.suggest_int("depth", 5, 9),
        "learning_rate":  trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
        "l2_leaf_reg":    trial.suggest_float("l2_leaf_reg", 2., 10., log=True),
        "random_strength":trial.suggest_float("random_strength", 1e-8, 1e-3, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 1.0),
        "iterations":     2000,
        "verbose":        False
    })
    cv = KFold(n_splits=3, shuffle=True, random_state=42)
    rmses = []
    for train_idx, val_idx in cv.split(X_train_fs):
        model = CatBoostRegressor(**params)
        model.fit(X_train_fs.iloc[train_idx], y_train_fs.iloc[train_idx],
                  eval_set=(X_train_fs.iloc[val_idx], y_train_fs.iloc[val_idx]),
                  verbose=False)
        preds = model.predict(X_train_fs.iloc[val_idx])
        rmses.append(mean_squared_error(y_train_fs.iloc[val_idx], preds, squared=False))
    return np.mean(rmses)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30, timeout=30*60)
best_params = study.best_trial.params
print("Optuna best:", best_params, " | RMSE‑cv:", study.best_value)


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
grid = {
    "depth":          [6, 7, 8],
    "learning_rate":  [0.03, 0.05, 0.08],
    "l2_leaf_reg":    [3, 5, 7],
    "bagging_temperature": [0, 0.5, 1],
}
grid_params = deepcopy(p["train"]["catboost"])
grid_params["iterations"] = 1500
grid_params["verbose"] = False

rs = RandomizedSearchCV(
    estimator=CatBoostRegressor(**grid_params),
    param_distributions=grid,
    n_iter=20,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    random_state=42,
)
rs.fit(X_train_fs, y_train_fs)
print("RandomSearch best:", rs.best_params_, " | RMSE‑cv:", -rs.best_score_)


In [ ]:
# See the project README for context.
best = best_params if study.best_value < -rs.best_score_ else rs.best_params_
final_params = deepcopy(p["train"]["catboost"])
final_params.update(best)
final_params["verbose"] = False
final_params["iterations"] = 2500

final_cb = CatBoostRegressor(**final_params)
final_cb.fit(X_train_fs, y_train_fs)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
preds = final_cb.predict(X_test_fs)
rmse_fin = mean_squared_error(y_test_fs, preds, squared=False)
mae_fin  = mean_absolute_error(y_test_fs, preds)
r2_fin   = r2_score(y_test_fs, preds)
print(f"FINAL v5 | RMSE: {rmse_fin:,.0f} | MAE: {mae_fin:,.0f} | R2: {r2_fin:.3f}")


In [ ]:
import mlflow, json, tempfile, pathlib

with mlflow.start_run(run_name="hyperopt_v5") as run:
    mlflow.log_params(final_params)
    mlflow.log_metrics({"rmse": rmse_fin, "mae": mae_fin, "r2": r2_fin})
    # See the project README for context.
    feat_path = pathlib.Path(tempfile.mkstemp(suffix=".json")[1])
    feat_path.write_text(json.dumps(best_feats, indent=2))
    mlflow.log_artifact(str(feat_path), artifact_path="selected_features")
    # See the project README for context.
    mlflow.catboost.log_model(
        final_cb,
        artifact_path="model",
        registered_model_name="real_estate_price_model"
    )
print("Version5 registered— run_id:", run.info.run_id)
